# SARIMA tuning (weekly SARIMA + trend + yearly Fourier exog)
Fit on the saved **train** split, score MAE on the saved **calibration** split (multi-step). Test is never touched here.
Best params go into `SARIMA_PARAMS` and `SARIMA_FOURIER_K` in `config.py`. Intervals are conformal (see `src/7_Sarima.py`), so selection is by MAE only.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import itertools, time, warnings
import numpy as np, pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from src.sarima_utils import fourier_terms, load_splits

warnings.filterwarnings("ignore")

In [2]:
train, calib, test = load_splits()
print(len(train), len(calib), len(test))   # earlier runs: 852 366 175
print(train.index[-1].date(), calib.index[[0, -1]].date)

# Reference: repeat the last 7 train days across the calib horizon
naive = np.tile(train.values[-7:], len(calib) // 7 + 1)[:len(calib)]
naive_mae = float(np.mean(np.abs(calib.values - naive)))
print("weekly seasonal-naive calib MAE:", round(naive_mae))

852 366 175
2023-12-31 [datetime.date(2024, 1, 1) datetime.date(2024, 12, 31)]
weekly seasonal-naive calib MAE: 8617


In [3]:
S = 7
K_GRID = [2, 4, 6]                                              # yearly Fourier harmonics
ARIMA_GRID = list(itertools.product(range(3), [0], range(2),    # p, d=0 (trend is in the exog), q
                                    range(2), range(2), range(2)))  # P, D, Q
print("fits:", len(K_GRID) * len(ARIMA_GRID))

rows = []
for K, (p, d, q, P, D, Q) in itertools.product(K_GRID, ARIMA_GRID):
    order, sorder = (p, d, q), (P, D, Q, S)
    t = time.time()
    try:
        res = SARIMAX(train, exog=fourier_terms(train.index, K),
                      order=order, seasonal_order=sorder,
                      enforce_stationarity=False,
                      enforce_invertibility=False).fit(disp=False, maxiter=200)
        pred = res.forecast(len(calib), exog=fourier_terms(calib.index, K)).values
        mae = float(np.mean(np.abs(calib.values - pred)))
    except Exception:
        mae = np.nan
    rows.append(dict(K=K, order=order, seasonal_order=sorder,
                     calib_mae=mae, secs=round(time.time() - t, 1)))

results = pd.DataFrame(rows).sort_values("calib_mae").reset_index(drop=True)
results.head(10)

fits: 144


,K,order,seasonal_order,calib_mae,secs
0,6,"(2, 0, 1)","(0, 1, 1, 7)",7704.344798,1.2
1,6,"(2, 0, 1)","(1, 1, 1, 7)",7742.937160,1.2
2,6,"(2, 0, 1)","(0, 1, 0, 7)",7773.089195,0.6
3,4,"(2, 0, 1)","(1, 1, 1, 7)",7796.641259,1.8
4,4,"(2, 0, 1)","(0, 1, 1, 7)",7826.628776,1.0
5,6,"(2, 0, 1)","(1, 1, 0, 7)",7894.217800,1.4
6,4,"(2, 0, 1)","(1, 1, 0, 7)",8004.576656,1.1
7,6,"(2, 0, 1)","(1, 0, 1, 7)",8064.787582,2.0
8,4,"(2, 0, 1)","(1, 0, 1, 7)",8098.586315,2.4
9,4,"(2, 0, 1)","(0, 1, 0, 7)",8124.790604,0.5


In [4]:
print("failed fits:", results["calib_mae"].isna().sum(), "of", len(results))
print("best MAE per K:")
print(results.groupby("K")["calib_mae"].min().round(0))

best = results.iloc[0]
print("\nbest calib MAE:", round(best.calib_mae), "| seasonal-naive:", round(naive_mae))
print()
print("# pasting into config.py")
print(f'SARIMA_PARAMS = {{"order": {best.order}, "seasonal_order": {best.seasonal_order}}}')
print(f"SARIMA_FOURIER_K = {best.K}")

failed fits: 0 of 144
best MAE per K:
K
2    9401.0
4    7797.0
6    7704.0
Name: calib_mae, dtype: float64

best calib MAE: 7704 | seasonal-naive: 8617

# pasting into config.py
SARIMA_PARAMS = {"order": (2, 0, 1), "seasonal_order": (0, 1, 1, 7)}
SARIMA_FOURIER_K = 6
